# 04 -- Model Training

**Purpose:** Train 11 regression models on the log-transformed CLV target, evaluate with R2 / RMSE / MAE / CV R2, and identify the best model for tuning.

| Step | Description |
|---|---|
| 1 | Load processed data + consensus features |
| 2 | Train-test split and scaling |
| 3 | Train 11 models |
| 4 | Compare results |
| 5 | Feature importance (best model) |

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.config import (PROCESSED_DATA_PATH, IMAGES_DIR, REPORTS_DIR,
                        TARGET_COL, TARGET_LOG_COL, RANDOM_STATE, TEST_SIZE)
from src.model import get_base_models, evaluate_model

pd.set_option('display.max_columns', None)

## 1. Load Data & Consensus Features

In [2]:
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [c for c in df.columns if c not in [TARGET_COL, TARGET_LOG_COL]]

# Load consensus feature set if available
consensus_path = PROCESSED_DATA_PATH.parent / 'consensus_features.json'
if consensus_path.exists():
    with open(consensus_path) as fp:
        consensus_list = json.load(fp)
    consensus_list = [f for f in consensus_list if f in feature_cols]
    print(f"Using consensus feature set: {len(consensus_list)} features")
else:
    consensus_list = feature_cols
    print(f"Using all features: {len(feature_cols)}")

X = df[consensus_list]
y = df[TARGET_LOG_COL]
print(f"\nDataset: {X.shape[0]:,} rows x {X.shape[1]} features")

Using consensus feature set: 50 features

Dataset: 9,134 rows x 50 features


## 2. Train-Test Split & Scaling

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

Train: (7307, 50)  Test: (1827, 50)


## 3. Train All 11 Models

In [4]:
models = get_base_models()
results = []
for name, model in models.items():
    print(f"  Training {name}...")
    metrics = evaluate_model(model, X_train_sc, X_test_sc, y_train, y_test)
    metrics['Model'] = name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index('Model').sort_values('R2', ascending=False)
print("\n--- Model Results (sorted by R2) ---")
print(results_df.to_string())

results_path = REPORTS_DIR / 'model_results.csv'
results_df.to_csv(results_path)
print(f"\nSaved -> {results_path}")

  Training Linear Regression...
  Training Ridge...
  Training Lasso...


  Training ElasticNet...


  Training Decision Tree...


  Training Random Forest...


  Training Gradient Boosting...


  Training Extra Trees...


  Training AdaBoost...


  Training SVR...


  Training XGBoost...



--- Model Results (sorted by R2) ---
                       R2    RMSE     MAE  CV_R2_Mean  CV_R2_Std
Model                                                           
Random Forest      0.9106  0.1989  0.0913      0.9093     0.0083
Extra Trees        0.9067  0.2032  0.0923      0.9079     0.0083
Gradient Boosting  0.9033  0.2068  0.1052      0.9017     0.0051
XGBoost            0.9006  0.2098  0.1089      0.9001     0.0090
AdaBoost           0.8515  0.2563  0.1857      0.8453     0.0172
Decision Tree      0.8373  0.2683  0.1084      0.8269     0.0149
SVR                0.4662  0.4861  0.3209      0.4325     0.0250
Linear Regression  0.3391  0.5409  0.4122      0.3307     0.0179
Ridge              0.3391  0.5409  0.4122      0.3307     0.0179
Lasso             -0.0018  0.6659  0.5333     -0.0001     0.0001
ElasticNet        -0.0018  0.6659  0.5333     -0.0001     0.0001

Saved -> D:\automation\Customer-Lifetime-Value-Prediction-For-AutoInsurance-Company\reports\model_results.csv


## 4. Compare Results

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# R2 comparison
results_df['R2'].sort_values().plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Model Comparison -- R2 Score', fontsize=13)
axes[0].set_xlabel('R2')
axes[0].axvline(0.9, color='red', linestyle='--', label='R2=0.9')
axes[0].legend()
for i, v in enumerate(results_df['R2'].sort_values()):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)

# RMSE comparison
results_df['RMSE'].sort_values(ascending=False).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Model Comparison -- RMSE', fontsize=13)
axes[1].set_xlabel('RMSE (lower is better)')
for i, v in enumerate(results_df['RMSE'].sort_values(ascending=False)):
    axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
fig.savefig(IMAGES_DIR / 'model_comparison_r2.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: model_comparison_r2.png")

Saved: model_comparison_r2.png


## 5. Feature Importance -- Best Model

In [6]:
from sklearn.ensemble import RandomForestRegressor

best_name = results_df.index[0]
print(f"Best model: {best_name} (R2={results_df.loc[best_name, 'R2']})")

# Refit best tree-based model for feature importance
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train_sc, y_train)

importances = pd.Series(rf.feature_importances_, index=consensus_list).nlargest(25)
fig, ax = plt.subplots(figsize=(10, 8))
importances.sort_values().plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('Top 25 Feature Importances (Random Forest)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: feature_importance.png")

Best model: Random Forest (R2=0.9106)


Saved: feature_importance.png


## Results Summary

| Metric | Target |
|---|---|
| R2 | > 0.85 |
| RMSE | Minimize on log scale |
| CV R2 | Close to test R2 (no overfitting) |

Top performers will be tuned in `05_Model_Evaluation.ipynb`.

---
**Next step: `05_Model_Evaluation.ipynb`**